In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from src.utils_jr import simulate_jr, alpha_delta_spectral_power, JRParams


t, eeg, states = simulate_jr(
    duration=20.0,
    fs=500.0,
    external_rate= 120,
    params=JRParams(
        A=3.9,
        B=21.0 # 21 to 29: 29 - maximum Delta, 21 - maximum Alpha
        )
)

power_times, alpha_power, delta_power = alpha_delta_spectral_power(
    t,
    eeg,
    sfreq=500.0,
    window_sec=4.0,
    step_sec=1.0,
)

fig, axes = plt.subplots(
    3, 1, figsize=(12, 10), sharex=True
)

axes[0].plot(t, eeg, linewidth=0.7)
axes[0].set_ylabel("JR output (mV)")
axes[0].set_title("Jansen-Rit output")

axes[1].plot(
    power_times, alpha_power, marker="o", markersize=3
)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel(r"Alpha power (mV$^2$)")
axes[1].set_title(f"Alpha spectral power; median {round(np.median(alpha_power), 3)}")

axes[2].plot(power_times, delta_power)
axes[2].set_ylabel(r"Delta power (mV$^2$")
axes[2].set_title(f"Delta spectral power; median {round(np.median(delta_power), 3)}")

for ax in axes:
    ax.grid(True, alpha=.3)

plt.tight_layout()
plt.show()

In [ ]:
def median_alpha_delta(ext_rate: float = 120, A: float = 3.25, B: float = 22.) -> tuple[float, float]:
    t, eeg, _ = simulate_jr(duration=20., fs=500.,
                            external_rate=ext_rate,
                            params=JRParams(A=A, B=B))
    _, alpha_pow, delta_pow = alpha_delta_spectral_power(t, eeg, sfreq=500., window_sec=4., step_sec=1.)
    return float(np.median(alpha_pow)), float(np.median(delta_pow))

median_alpha_delta()

In [ ]:
from itertools import product
from tqdm import tqdm


def grid_search_jr(
    A_values: np.ndarray,
    B_values: np.ndarray,
    ext_rate: float = 120.0,
    min_alpha_pow: float = 1e-4,
    min_delta_pow: float = 1e-4,
) -> list[dict]:
    results = []

    for A, B in tqdm(list(product(A_values, B_values))):
        alpha_pow, delta_pow = median_alpha_delta(
            ext_rate=ext_rate,
            A=float(A),
            B=float(B),
        )

        # Reject trials with insufficient absolute power.
        if (
            not np.isfinite(alpha_pow)
            or not np.isfinite(delta_pow)
            or alpha_pow < min_alpha_pow
            or delta_pow < min_delta_pow
        ):
            continue

        score = delta_pow / alpha_pow

        result = {
            "A": float(A),
            "B": float(B),
            "alpha_pow": alpha_pow,
            "delta_pow": delta_pow,
            "ratio": score,
        }
        results.append(result)

    results.sort(key=lambda x: x["ratio"], reverse=True)
    return results


A_values = np.linspace(2.5, 4.0, 16)
B_values = np.linspace(15.0, 30.0, 16)

results = grid_search_jr(
    A_values=A_values,
    B_values=B_values,
    ext_rate=120.0,
    min_alpha_pow=1e-4,
    min_delta_pow=1e-4,
)

print("\nTop 10 valid trials:")
for result in results[:10]:
    print(result)